# Build an Object Detection Assistant

Companion notebook for the To Data & Beyond tutorial. The application detects everyday objects with Meta's DETR model, draws labeled bounding boxes, summarizes the detections in natural language, speaks the summary with a text-to-speech model, and exposes the visual pipeline through Gradio.

> A GPU is optional but makes inference faster. Model weights are downloaded from the public Hugging Face Hub; no API keys are required. The Gradio public-share URL is temporary.

## 1. Install and import the dependencies

In [ ]:
%pip install -q transformers gradio timm inflect phonemizer matplotlib requests pillow

In [ ]:
import io
from collections import Counter

import gradio as gr
import inflect
import matplotlib.pyplot as plt
import requests
import torch
from IPython.display import Audio as IPythonAudio
from PIL import Image
from transformers import pipeline

## 2. Define the helper functions

In [ ]:
def load_image_from_url(url: str) -> Image.Image:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(io.BytesIO(response.content)).convert("RGB")

In [ ]:
def render_results_in_image(in_pil_img, in_results):
    plt.figure(figsize=(16, 10))
    plt.imshow(in_pil_img)
    ax = plt.gca()

    for prediction in in_results:
        x = prediction["box"]["xmin"]
        y = prediction["box"]["ymin"]
        width = prediction["box"]["xmax"] - x
        height = prediction["box"]["ymax"] - y
        ax.add_patch(
            plt.Rectangle(
                (x, y), width, height, fill=False, color="green", linewidth=2
            )
        )
        ax.text(
            x,
            y,
            f"{prediction['label']}: {prediction['score'] * 100:.1f}%",
            color="red",
            bbox={"facecolor": "white", "alpha": 0.75, "edgecolor": "none"},
        )

    plt.axis("off")
    image_buffer = io.BytesIO()
    plt.savefig(image_buffer, format="png", bbox_inches="tight", pad_inches=0)
    image_buffer.seek(0)
    modified_image = Image.open(image_buffer).copy()
    plt.close()
    return modified_image

In [ ]:
def summarize_predictions_natural_language(predictions):
    word_engine = inflect.engine()
    counts = Counter(prediction["label"] for prediction in predictions)
    phrases = []

    for label, count in counts.items():
        noun = word_engine.plural_noun(label, count) or label
        phrases.append(f"{word_engine.number_to_words(count)} {noun}")

    if not phrases:
        return "No objects were detected in this image."
    return f"In this image, there are {word_engine.join(phrases)}."

## 3. Build the object-detection pipeline

In [ ]:
device = 0 if torch.cuda.is_available() else -1
od_pipe = pipeline(
    "object-detection",
    model="facebook/detr-resnet-50",
    device=device,
)

In [ ]:
sample_image_url = (
    "https://cdn-images-1.medium.com/max/1200/"
    "1*LjNoX7z5-M5vt3RbziGl6A.png"
)
raw_image = load_image_from_url(sample_image_url)
raw_image.resize((569, 491))

In [ ]:
pipeline_output = od_pipe(raw_image)
processed_image = render_results_in_image(raw_image, pipeline_output)
processed_image

## 4. Build the Gradio application

In [ ]:
def get_pipeline_prediction(pil_image):
    detections = od_pipe(pil_image)
    return render_results_in_image(pil_image, detections)


demo = gr.Interface(
    fn=get_pipeline_prediction,
    inputs=gr.Image(label="Input image", type="pil"),
    outputs=gr.Image(label="Output image with predicted instances", type="pil"),
    title="Object Detection Assistant",
)

In [ ]:
demo.launch(share=True)

In [ ]:
demo.close()

## 5. Turn the detections into a spoken description

In [ ]:
pipeline_output

In [ ]:
text = summarize_predictions_natural_language(pipeline_output)
text

In [ ]:
tts_pipe = pipeline(
    "text-to-speech",
    model="kakao-enterprise/vits-ljs",
)
narrated_text = tts_pipe(text)

In [ ]:
IPythonAudio(
    narrated_text["audio"],
    rate=narrated_text["sampling_rate"],
)